# Tutorial: Task Notebook

Audience:
- Anyone who wants a repeatable notebook pattern for a small analysis task.

Prerequisites:
- Basic Python and notebook usage.

Learning goals:
- Build a tiny dataset in a deterministic way.
- Compute and interpret a few practical summary metrics.
- Test one simple what-if change and compare outcomes.


## Outline

1. Setup
2. Create a small dataset
3. Baseline summary metrics
4. What-if scenario
5. Exercise and concise takeaways


In [ ]:
# Setup: keep state deterministic and imports minimal.
from __future__ import annotations

import random
from statistics import mean, median

SEED = 42
random.seed(SEED)
SEED


## Step 1 - Create a small dataset

Generate synthetic task durations (in minutes). The list is small on purpose so the logic stays easy to inspect.


In [ ]:
task_durations = [random.randint(8, 45) for _ in range(20)]
task_durations


## Step 2 - Compute baseline metrics

Use a compact function so the summary can be reused for baseline and scenario comparisons.


In [ ]:
def summarize(values: list[int]) -> dict[str, float]:
    ordered = sorted(values)
    p90_index = int(0.9 * (len(ordered) - 1))
    return {
        "count": float(len(values)),
        "mean": round(mean(values), 2),
        "median": float(median(values)),
        "p90": float(ordered[p90_index]),
        "max": float(max(values)),
    }

baseline = summarize(task_durations)
baseline


## Step 3 - Run one what-if scenario

Assume the slowest 25% of tasks get 10% faster. Then compare before vs after.


In [ ]:
sorted_values = sorted(task_durations)
cutoff = int(len(sorted_values) * 0.75)
threshold = sorted_values[cutoff]

improved = [int(v * 0.9) if v >= threshold else v for v in task_durations]
scenario = summarize(improved)

{
    "baseline": baseline,
    "scenario": scenario,
    "delta_mean": round(scenario["mean"] - baseline["mean"], 2),
    "delta_p90": round(scenario["p90"] - baseline["p90"], 2),
}


## Exercises

- Change the scenario from `10%` to `15%` for the slowest tasks.
- Compare how `mean` and `p90` respond.
- Note which metric moves more and why.


In [ ]:
def run_scenario(values: list[int], speedup: float, tail_share: float = 0.25) -> dict[str, dict[str, float]]:
    ordered = sorted(values)
    threshold = ordered[int(len(ordered) * (1 - tail_share))]
    changed = [int(v * (1 - speedup)) if v >= threshold else v for v in values]
    return {"before": summarize(values), "after": summarize(changed)}

run_scenario(task_durations, speedup=0.15)


## Concise Takeaways

- Keep setup deterministic (`seed`) so results are reproducible.
- A tiny reusable summary function makes comparisons fast.
- Scenario testing is most useful when it targets a clear bottleneck segment.
